# 01 · Auditoria e qualidade dos dados

**Projeto:** Customer Retention Analytics  
**Janela:** junho–dezembro de 2025  
**Fonte:** planilha privada do case, fora do controle de versão  
**Unidade provisória:** registro da planilha / possível episódio de atendimento  
**Objetivo:** verificar a qualidade do dado e a observabilidade dos desfechos antes de identificar perfis ou modelar.

> As flags e os status têm interpretação **provisória**. Um registro `RETIDO` não comprova permanência do cliente no futuro, e o valor monetário não foi validado como receita recorrente.

Execute antes, na raiz do projeto: `retention-audit --input data/raw/retention_case.xlsx --output reports/private/audit_summary.json`. O resumo será reconstruído a cada nova versão da fonte; o notebook não lê registros individuais.

## TL;DR — preenchido por execução

Este notebook só exibe resultados carregados de um arquivo de auditoria **local**, produzido com fingerprint e fonte autorizada. As cifras não são incluídas na versão controlada no GitHub. As conclusões apresentadas são **descritivas**, não causais.

Os campos M0 e M30 representam estados, não necessariamente intervalos individuais de 30 dias.

## 1. Preparação e proveniência

Usa exclusivamente os agregados existentes em `reports/private/audit_summary.json`. O arquivo contém SHA-256 da fonte e regras de recorte e permanece ignorado pelo Git.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt

REPORT = Path("reports/private/audit_summary.json")
assert REPORT.is_file(), "Execute retention-audit a partir da raiz do projeto."

with REPORT.open(encoding="utf-8") as handle:
    audit = json.load(handle)

assert audit["period"] == {"start": "2025-06", "end": "2025-12"}
assert audit["definitions_status"] == "PROVISIONAL_UNVALIDATED"
assert audit["overall"]["rows_in_scope"] == sum(audit["monthly_rows"].values())

print("Registros lidos:", audit["overall"]["rows_total"])
print("Dentro do escopo:", audit["overall"]["rows_in_scope"])
print("Fingerprint SHA-256:", audit["provenance"]["source_sha256"])
print("Status de definições:", audit["definitions_status"])

## 2. Dimensões e denominadores

- **Volume do funil:** registros com data de entrada válida na janela, sem deduplicação por cliente.
- **Retenção observada:** categoria literal `RETIDO` entre status observados; pendentes mantidos fora do denominador de leads concluídos.
- **Retenção com movimentação:** status `RETIDO` e ao menos uma `FLAG_MOV_* = 1`.
- **Percentual de movimentação entre retidos:** retidos com movimentação / total de retidos.
- **Valor monetário:** apenas resumo dos campos, sem inferir faturamento real ou benefício financeiro.

A taxa de clientes ativos e churn sustentado **não é computável** com esta fonte sem histórico contratual posterior.

In [ ]:
total = audit["overall"]["rows_in_scope"]
retained = audit["status_in_scope"].get("RETIDO", 0)
pending = audit["status_in_scope"].get("LEAD EM ABERTO", 0)
retained_mov = audit["overall"].get("retained_movement", 0)

summary = {
    "registros_elegiveis": total,
    "retidos": retained,
    "pendentes": pending,
    "retidos_com_movimentacao": retained_mov,
    "movimentacao_entre_retidos": retained_mov / retained if retained else None,
    "ids_distintos_escopo": audit["clients"]["distinct_in_scope"],
}
for label, value in summary.items():
    print(f"{label}: {value:.3%}" if isinstance(value, float) else f"{label}: {value:,}")

## 3. Cobertura temporal e processo

Uma amplitude semelhante de volumes por mês não prova amostragem aleatória nem representatividade temporal. As comparações SOLAR × LEGADO devem considerar a composição de status pendentes.

In [ ]:
periods = list(audit["monthly_rows"])
volumes = [audit["monthly_rows"][month] for month in periods]
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.bar(periods, volumes, color="#315f5e")
ax.set_title("Registros elegíveis por mês de entrada")
ax.set_xlabel("Mês de entrada")
ax.set_ylabel("Registros (amostra)")
ax.grid(axis="y", alpha=0.18)
plt.tight_layout()
plt.show()

## 4. Movimentação condicional à retenção

Este indicador compara episódios **classificados como retidos**. Ele não é taxa de aceite de ofertas e não deve ser interpretado como efeito de SOLAR ou LEGADO.

In [ ]:
by_month = [row for row in audit["segment_profiles"] if row["dimension"] == "month"]
by_month.sort(key=lambda x: x["category"])
months = [row["category"] for row in by_month]
rates = [100 * row["retained_movement_given_retained"] for row in by_month]
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.plot(months, rates, "o-", linewidth=2, color="#315f5e")
ax.set_title("Movimentação entre retidos por mês de entrada")
ax.set_xlabel("Mês")
ax.set_ylabel("% de registros retidos com movimentação")
ax.set_ylim(0, 100)
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()
print("Base do gráfico: somente registros retidos em cada mês.")

## 5. Transições de portfólio e reconciliação

As mudanças de código M0→M30 são registradas separadamente das flags operacionais. Um mesmo código com valor diferente não configura automaticamente movimentação contratual. Divergências entre flags e transições devem ser explicadas antes da criação do alvo final.

In [ ]:
for family, values in audit["products"].items():
    print(f"{family}: M0={values.get('present_m0', 0):,}, "
          f"M30={values.get('present_m30', 0):,}, "
          f"FLAG_MOV={values.get('flag_movement', 0):,}, "
          f"FLAG_CANC={values.get('flag_cancellation', 0):,}, "
          f"troca_sem_flag={values.get('changed_code_without_flag', 0):,}")

## 6. Qualidade e cobertura

Missingness em campo de produto pode ser ausência estrutural quando o cliente não possui aquela família. O diagnóstico informa a frequência; não realiza imputações automáticas.

In [ ]:
missing = sorted(audit["missing_all_rows"].items(), key=lambda x: x[1], reverse=True)
print("Campos mais ausentes na planilha integral:")
for name, count in missing[:12]:
    print(f"{name}: {count:,} ({count / audit['overall']['rows_total']:.1%})")
print("IDs com mais de um episódio candidato:", audit["clients"]["repeated_ids_in_scope"])
print("IDs com status diferentes:", audit["clients"]["ids_with_mixed_status"])

## 7. Interpretação e decisões pendentes

**Fatos verificáveis localmente:** integridade da leitura, quantidades por mês/status, IDs repetidos, flags e variações de código/valor.  
**Inferências sujeitas a validação:** classificação de um registro como retenção de relacionamento, significado financeiro da diferença M0/M30 e equivalência das flags aos eventos contratuais.  
**Hipóteses de negócio a testar:** perfis iniciais associados a maior movimentação entre os retidos, inclusive efeitos de composição por origem e família inicial.

Não há data de churn posterior que autorize Kaplan–Meier/Cox sobre o relacionamento nem base causal que autorize estimativa incremental de oferta.  

**Próximo gate:** validar a chave de episódio, os estados de funil, a semântica de movimentação e a natureza dos campos VALOR com a área responsável. Somente então evoluir para perfis inferenciais, baseline e validação temporal.

As contagens deste notebook são por registros da amostra. Não divulgá-las fora do ambiente autorizado sem avaliação de confidencialidade.